In [67]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
import time
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
# macbook : "mps"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [68]:
import os

with open("mov.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"Total characters: {len(text):,}")
print(f"First 200 characters:\n{text[:200]}")

Total characters: 130,853
First 200 characters:
SCENE:
Venice, and PORTIA'S house at Belmont


ACT 1. SCENE I.
Venice. A street

Enter ANTONIO, SALERIO, and SOLANIO

  ANTONIO. In sooth, I know not why I am so sad.
    It wearies me; you say it wea


In [69]:
# built character vocubulary
chars=sorted(set(text))
vocab_size=len(chars)

# charater to index and vice versa
char_to_idx={c:i for i,c in enumerate(chars)}
idx_to_char={i:c for i,c in enumerate(chars)}

#encode and decode function for inputs
def encode(s):
    return[char_to_idx[c] for c in s]
def decode(ids):
    return "".join([idx_to_char[i] for i in ids])

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars)}")
print(f"\nExample encoding:")
print(f"  'Antonio' -> {encode('Antonio')}")
print(f"  {encode('Antonio')} -> '{decode(encode('Antonio'))}'")

Vocabulary size: 78
Characters: 
 !'(),-.0123456789:;<>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]abcdefghijklmnopqrstuvwxyz

Example encoding:
  'Antonio' -> [24, 65, 71, 66, 65, 60, 66]
  [24, 65, 71, 66, 65, 60, 66] -> 'Antonio'


In [70]:
#split the data into two parts, 85% for model learning and 15% for model validaion
data=torch.tensor(encode(text), dtype=torch.long)
n=int(0.85*len(data))
train_data=data[:n]
val_data=data[n:]
print(f"Training token: {len(train_data):,}")
print(f"Validation token: {len(val_data):,}")

Training token: 111,225
Validation token: 19,628


In [71]:
# Batching: grab random chunks of text
def get_batch(split, batch_size, context_length):
    d=train_data if split=="train" else val_data
    ix=torch.randint(len(d)-context_length, (batch_size,))
    x=torch.stack([d[i:i+context_length] for i in ix])
    y=torch.stack([d[i+1:i+context_length+1] for i in ix])
    return x.to(device),y.to(device)

# Quick test of batching
xb, yb = get_batch("train", batch_size=4, context_length=15)
print(f"Input shape:  {xb.shape}  (batch_size x context_length)")
print(f"Target shape: {yb.shape}")
print(f"\nExample (first sequence):")
print(f"  Input:  {decode(xb[0].tolist())!r}")
print(f"  Target: {decode(yb[0].tolist())!r}")
print(f"  (Target is input shifted by 1 character)")

Input shape:  torch.Size([4, 15])  (batch_size x context_length)
Target shape: torch.Size([4, 15])

Example (first sequence):
  Input:  'se, in love I d'
  Target: 'e, in love I do'
  (Target is input shifted by 1 character)


# **RMSNorm**
RMSNorm skips the mean subtraction and the shift as of in layerNorm. Just divide by the root-mean-square, then scale. Simpler, faster, works just as well.

**Root Mean Square Layer Normalization.**

Simpler than LayerNorm:
- No mean subtraction
- No bias/shift parameter
- Just: x / RMS(x) * learnable_scale

**DROPOUT**
- We will also use Dropout throughout the model.
- Dropout randomly zeroes some values during training,
- forcing the model to not rely on any single feature.
- This prevents memorization (overfitting).

In [72]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps=eps
        self.weight=nn.Parameter(torch.ones(dim))

    def forward(self,x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.weight
DROPOUT=0.2

# --- Demo ---
demo_x = 1000 + (2000 - 1000) * torch.rand(2, 4, 10)
norm = RMSNorm(10)
demo_out = norm(demo_x)

print("RMSNorm demo:")
print(f"  Input  - mean: {demo_x.mean():.3f}, std: {demo_x.std():.3f}")
print(f"  Output - mean: {demo_out.mean():.3f}, std: {demo_out.std():.3f}")
print(f"  Input range:  [{demo_x.min():.3f}, {demo_x.max():.3f}]")
print(f"  Output range: [{demo_out.min():.3f}, {demo_out.max():.3f}]")
print(f"\n  Parameters: just a scale vector of size {norm.weight.shape}")

RMSNorm demo:
  Input  - mean: 1493.622, std: 259.736
  Output - mean: 0.986, std: 0.169
  Input range:  [1001.666, 1973.453]
  Output range: [0.681, 1.313]

  Parameters: just a scale vector of size torch.Size([10])


# **RoPE**
RoPE rotates Q and K vectors based on position. The dot product then naturally captures relative distance between tokens.

Like a clock: the angle between 3 o'clock and 5 o'clock = the angle between 7 o'clock and 9 o'clock. Absolute position doesn't matter, only the gap.

In [73]:
#precompute give different frequencies to each pair and produce angle according to them 
def precompute_rope_freqs(head_dim, max_seq_len, base=10000.0):
    freqs = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    positions = torch.arange(max_seq_len).float()
    angles = torch.outer(positions, freqs)
    return torch.cos(angles), torch.sin(angles)

#applying RoPE
def apply_rope(x, cos, sin):
    seq_len = x.shape[2]
    cos = cos[:seq_len].unsqueeze(0).unsqueeze(0)  # [1, 1, seq, hd//2]
    sin = sin[:seq_len].unsqueeze(0).unsqueeze(0)
    x1 = x[..., ::2]   # even dims
    x2 = x[..., 1::2]  # odd dims
    out1 = x1 * cos - x2 * sin
    out2 = x1 * sin + x2 * cos
    return torch.stack([out1, out2], dim=-1).flatten(-2)

# **GQA**
GQA keeps many Query heads but makes groups of Query heads share fewer Key/Value heads, while every K/V head still contains information across all tokens.reducing K/V heads doesn't mean reducing the number of tokens or automatically making the model unable to find the right answer.

In [74]:
#repear_kv is used to copy key,value. like if we have K1 if n_rep=3 it will become K1 K1 K1
def repeat_kv(x, n_rep):
    if(n_rep==1):
        return x
    b,n_kv,seq,hd=x.shape
    return(x[:,:,None,:,:].expand(b,n_kv,n_rep,seq,hd).reshape(b,n_kv*n_rep,seq,hd))

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        assert d_model%n_heads==0
        assert n_heads%n_kv_heads==0

        #we use self. because we want that value in further program
        self.n_heads=n_heads
        self.n_kv_heads=n_kv_heads
        self.n_rep=n_heads//n_kv_heads
        self.head_dim=d_model//n_heads

        #creating lyers for query,key,value,output
        self.q_proj=nn.Linear(d_model, n_heads*self.head_dim, bias=False)
        self.k_proj=nn.Linear(d_model, n_kv_heads*self.head_dim, bias=False)
        self.v_proj=nn.Linear(d_model, n_kv_heads*self.head_dim, bias=False)
        self.o_proj=nn.Linear(n_heads*self.head_dim, d_model, bias=False)

    def forward(self, x, rope_cos, rope_sin):
        b, seq, _ = x.shape

        # Project Q, K, V
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # Reshape into heads
        q = q.view(b, seq, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(b, seq, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = v.view(b, seq, self.n_kv_heads, self.head_dim).transpose(1, 2)

        # Apply RoPE to Q and K (not V!)
        q = apply_rope(q, rope_cos, rope_sin)
        k = apply_rope(k, rope_cos, rope_sin)

        # Repeat KV heads to match Q heads
        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        # Scaled dot-product attention with causal mask
        scale = 1.0 / math.sqrt(self.head_dim)
        scores = (q @ k.transpose(-2, -1)) * scale

        mask = torch.triu(torch.ones(seq, seq, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)

        # Dropout on attention weights (regularization)
        weights = F.dropout(weights, p=DROPOUT, training=self.training)

        out = weights @ v

        # Merge heads and project
        out = out.transpose(1, 2).contiguous().view(b, seq, -1)
        return self.o_proj(out)

# **SILU**

SwiGLU splits into two paths — a gate that decides how much info flows through, and a value path that carries the info. They combine via element-wise multiplication.

    SwiGLU Feed-Forward Network.

    Two paths:
      gate: SiLU(x @ W_gate) - controls flow
      up:   x @ W_up         - carries information

    Combined: gate * up -> W_down

    SiLU(x) = x * sigmoid(x), a smooth version of ReLU.


In [75]:
class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_dim):
        super().__init__()
        self.w_gate = nn.Linear(d_model, hidden_dim, bias=False)
        self.w_up   = nn.Linear(d_model, hidden_dim, bias=False)
        self.w_down = nn.Linear(hidden_dim, d_model, bias=False)

    def forward(self, x):
        gate = F.silu(self.w_gate(x))
        up   = self.w_up(x)
        return F.dropout(self.w_down(gate * up), p=DROPOUT, training=self.training)

# **Transformer Block**

it puts all the layers together into one block

One layer of a modern transformer.

    Pre-norm architecture:
      x -> RMSNorm -> GQA Attention -> + residual
      x -> RMSNorm -> SwiGLU FFN     -> + residual

In [76]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, ffn_hidden_dim):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attention = GroupedQueryAttention(d_model, n_heads, n_kv_heads)
        self.ffn_norm  = RMSNorm(d_model)
        self.ffn       = SwiGLU(d_model, ffn_hidden_dim)
    def forward(self, x, rope_cos, rope_sin):
        x = x + self.attention(self.attn_norm(x), rope_cos, rope_sin)
        x = x + self.ffn(self.ffn_norm(x))
        return x    

# **Creating LLM**
first we will call aall the required data then the proces goes in the following way:-
1. we will create token emmbiding 

2. then pass than emmbiding to the no of transformer blocks

3. the result fter going through all the transformer blocks is again normalized using rms 

4. then we see that for e.g.- if w have 65 charactersin total we get a 768 sized embbiding so using lm head we convert that 768 to a 65 sized score one for each character

5. Weight Tying: Using the same weight matrix for the Token Embedding and LM Head to reduce parameters and share the learned token representations.

6. RoPE Precomputation: We calculate and store the cosine and sine values required for RoPE beforehand for every position and dimension; they are registered as non-learnable buffers so they move with the model but are not updated during training.

7. Weight initialization gives the model's learnable weights small random starting values before training, rather than keeping it zero.

8. here forward() function takes the input characters (idx), converts each character ID into a vector using the token embedding, sends those vectors through all Transformer blocks so that the characters can understand the context around them, normalizes the resulting representations, converts each representation into scores for every possible character using the LM head, and, if the correct answers (targets) are provided, compares the model's predictions with those correct answers to calculate the loss. Finally, it returns the predictions (logits) and the loss.

In [77]:
class MiniLLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, ffn_hidden_dim, max_seq_len):
        super().__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        #token emmbedding
        self.token_emb = nn.Embedding(vocab_size, d_model)

        #creating transformer block
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, n_kv_heads, ffn_hidden_dim)
            for _ in range(n_layers)
        ])

        # Final norm and output head
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: share embedding and output weights
        self.lm_head.weight = self.token_emb.weight

        # Precompute RoPE frequencies
        head_dim = d_model // n_heads
        rope_cos, rope_sin = precompute_rope_freqs(head_dim, max_seq_len)
        self.register_buffer("rope_cos", rope_cos)
        self.register_buffer("rope_sin", rope_sin)

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        b, seq_len = idx.shape

        # Token embedding
        x = self.token_emb(idx)

        # Pass through transformer blocks
        for layer in self.layers:
            x = layer(x, self.rope_cos, self.rope_sin)

        # Final norm + project to vocabulary
        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )

        return logits, loss
        



In [78]:
# --- Model Configuration ---

config = {
    "vocab_size":     vocab_size,
    "d_model":        256,
    "n_layers":       4,
    "n_heads":        8,
    "n_kv_heads":     2,
    "ffn_hidden_dim": 680,
    "max_seq_len":    256,
}

model = MiniLLM(**config).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 50)
print("  MODEL SUMMARY")
print("=" * 50)
print(f"  Vocabulary:      {config['vocab_size']}")
print(f"  Embedding dim:   {config['d_model']}")
print(f"  Layers:          {config['n_layers']}")
print(f"  Query heads:     {config['n_heads']}")
print(f"  KV heads:        {config['n_kv_heads']} (GQA ratio: {config['n_heads']//config['n_kv_heads']}:1)")
print(f"  FFN hidden dim:  {config['ffn_hidden_dim']}")
print(f"  Context length:  {config['max_seq_len']}")
print(f"  Head dim:        {config['d_model'] // config['n_heads']}")
print(f"{'=' * 50}")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size (approx):  {total_params * 4 / 1e6:.1f} MB (float32)")
print(f"{'=' * 50}")

  MODEL SUMMARY
  Vocabulary:      78
  Embedding dim:   256
  Layers:          4
  Query heads:     8
  KV heads:        2 (GQA ratio: 4:1)
  FFN hidden dim:  680
  Context length:  256
  Head dim:        32
  Total parameters:     2,766,592
  Trainable parameters: 2,766,592
  Model size (approx):  11.1 MB (float32)


In [79]:
# --- Training Hyperparameters ---
BATCH_SIZE = 64
CONTEXT_LEN = config["max_seq_len"]
LEARNING_RATE = 3e-4
MAX_STEPS = 3000
EVAL_INTERVAL = 250
EVAL_STEPS = 20
LOG_INTERVAL = 50

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# **Model Checking**

estimate_loss() temporarily puts the model into evaluation mode, takes 20 batches from both the training and validation datasets, calculates the loss for each batch without calculating gradients, averages those losses, and returns the average train and validation loss before switching the model back to training mode.

In [80]:
@torch.no_grad()
def estimate_loss():
    """Estimate loss on train and val splits."""
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(EVAL_STEPS):
            xb, yb = get_batch(split, BATCH_SIZE, CONTEXT_LEN)
            _, loss = model(xb, yb)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

# **Training**
During training, the model first takes a random batch of input text (xb) and their correct next characters (yb). It passes the input through the Transformer and produces predictions (logits) and calculates the loss, which tells how wrong the predictions are. Then optimizer.zero_grad() clears the previous gradients, loss.backward() calculates how each model parameter should change to reduce the loss, and clip_grad_norm_() prevents excessively large updates. Finally, optimizer.step() uses those gradients to update the model's parameters, making the model slightly better. This process repeats for 3000 steps. Every 50 steps the current loss is printed, and every 250 steps the model is evaluated on both training and validation data to track how well it is learning.

In [84]:
# --- Training Loop ---
print("Starting training...")
print(f"  {MAX_STEPS} steps, batch_size={BATCH_SIZE}, context_len={CONTEXT_LEN}")
print(f"  Evaluating every {EVAL_INTERVAL} steps")
print("-" * 60)

train_losses = []
val_losses = []
step_log = []
start_time = time.time()

model.train()
for step in range(MAX_STEPS):
    xb, yb = get_batch("train", BATCH_SIZE, CONTEXT_LEN)

    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if step % LOG_INTERVAL == 0:
        elapsed = time.time() - start_time
        print(f"  Step {step:5d}/{MAX_STEPS} | Loss: {loss.item():.4f} | Time: {elapsed:.0f}s")

    if step % EVAL_INTERVAL == 0 or step == MAX_STEPS - 1:
        losses = estimate_loss()
        train_losses.append(losses["train"])
        val_losses.append(losses["val"])
        step_log.append(step)
        if step > 0:
            elapsed = time.time() - start_time
            steps_per_sec = step / elapsed
            remaining = (MAX_STEPS - step) / steps_per_sec
            print(f"  >>> Eval @ step {step}: train={losses['train']:.4f}, val={losses['val']:.4f} | ~{remaining:.0f}s remaining")

total_time = time.time() - start_time
print("-" * 60)
print(f"Training complete! Total time: {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Final val loss:   {val_losses[-1]:.4f}")

Starting training...
  3000 steps, batch_size=64, context_len=256
  Evaluating every 250 steps
------------------------------------------------------------
  Step     0/3000 | Loss: 3.3816 | Time: 0s
  Step    50/3000 | Loss: 2.4064 | Time: 14s
  Step   100/3000 | Loss: 2.0225 | Time: 25s
  Step   150/3000 | Loss: 1.7294 | Time: 36s
  Step   200/3000 | Loss: 1.5651 | Time: 47s
  Step   250/3000 | Loss: 1.4162 | Time: 58s
  >>> Eval @ step 250: train=1.3161, val=1.4516 | ~677s remaining
  Step   300/3000 | Loss: 1.3520 | Time: 72s
  Step   350/3000 | Loss: 1.2479 | Time: 83s
  Step   400/3000 | Loss: 1.1815 | Time: 94s
  Step   450/3000 | Loss: 1.1071 | Time: 105s
  Step   500/3000 | Loss: 1.0377 | Time: 117s
  >>> Eval @ step 500: train=0.8189, val=1.4837 | ~598s remaining
  Step   550/3000 | Loss: 0.9476 | Time: 131s
  Step   600/3000 | Loss: 0.9036 | Time: 142s
  Step   650/3000 | Loss: 0.8114 | Time: 153s
  Step   700/3000 | Loss: 0.7469 | Time: 164s
  Step   750/3000 | Loss: 0.7129

# **Generating Text**

In [89]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=700, temperature=0.8):
    """
    Generate text autoregressively.

    temperature controls randomness:
      low (0.3)  -> conservative, repetitive
      mid (0.8)  -> balanced
      high (1.2) -> creative, chaotic
    """
    model.eval()
    tokens = encode(prompt)
    tokens = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        context = tokens[:, -config["max_seq_len"]:]
        logits, _ = model(context)
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        tokens = torch.cat([tokens, next_token], dim=1)

    return decode(tokens[0].tolist())

In [100]:
# --- Generate at different temperatures ---
prompt = input("Enter your prompt: ")

print("=" * 60)
print(f"  PROMPT: {prompt!r}")
print("=" * 60)

for temp in [0.5, 0.8, 1.0, 1.2]:
    print(f"\n{'_' * 60}")
    print(f"  Temperature = {temp}")
    print(f"{'_' * 60}")
    output = generate(model, prompt, max_new_tokens=300, temperature=temp)
    print(output)

  PROMPT: 'Morocco:'

____________________________________________________________
  Temperature = 0.5
____________________________________________________________
Morocco:
    The part and man in no toward doth sit the sepulchre.
    Thus ornament is but the guiled shore
    To a most dangerous sea; the beauteous scarf
    Veiling an Indian beauty; in a word,
    The seeming truth which cunning times put on
    To entrap the wisest. Therefore, thou gaudy gold,
    Ha

____________________________________________________________
  Temperature = 0.8
____________________________________________________________
Morocco:
    I have a pound of flesh, to be by him cut off
    Nearest the merchant's heart. Be merciful.
    Take thrice thy money; bid me tear the bond.
  SHYLOCK. When it is paid according to the tenour.
    It doth appear you are a worthy judge;
    You know the law; your exposition
    Hath been most s

____________________________________________________________
  Temperature